# 📰 News Embedding Worker
Embeds news articles using **Qwen3-Embedding-8B** on Kaggle T4 GPU and saves vectors to PostgreSQL via n8n webhooks.

---
### ⚠️ Before running this notebook:
1. Attach the `qwen3-embedding-8b-model` dataset (right panel → Input → Add Input)
2. Set Accelerator to **GPU T4 x2** (right panel)
3. Turn Internet **ON** (right panel)
4. Fill in your credentials in **Cell 2** below
5. Run all cells top to bottom: **Cell 1 → 2 → 3 → 4**
---

In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# Run once per session. Takes ~1 minute.
# ============================================================
import subprocess, sys, os

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "sentence-transformers", "psutil", "accelerate"])

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("✅ Dependencies installed")

---
## ✏️ Cell 2 — Fill in your credentials here

In [ ]:
# ============================================================
# CELL 2 — Configuration
# Edit the values below before running.
# ============================================================
import os
import glob
import socket
import platform
import psutil
from collections import deque
from datetime import datetime, timezone

os.environ["PYTORCH_ALLOC_CONF"] = "max_split_size_mb:128,garbage_collection_threshold:0.6,expandable_segments:True"

# --- n8n webhook URLs (get these from your n8n instance) ---
N8N_GET_URL  = "https://n8n.3rfan.ir/webhook/4f1d52bf-25d5-4e0b-ab30-123f680d0265"     # e.g. https://n8n.yourdomain.com/webhook/xxx
N8N_SAVE_URL = "https://n8n.3rfan.ir/webhook/4f1d52bf-25d5-4e0b-ab30-123f680d0255"  # e.g. https://n8n.yourdomain.com/webhook/yyy
N8N_STATUS_URL = "https://n8n.3rfan.ir/webhook/worker-status"                    # For sending progress updates / heartbeat; set to None to disable
N8N_API_KEY  = "mer30kehasti"               # the X-API-Key secret set in n8n
N8N_HEADERS  = {"X-API-Key": N8N_API_KEY, "Content-Type": "application/json"}

# --- Worker identity (use a unique name if running multiple workers) ---
NODE_NAME    = "kaggle-t4-worker"
MODEL_NAME   = "Qwen/Qwen3-Embedding-8B"

# --- Performance settings ---
BATCH_SIZE   = 1      # articles fetched per cycle (keep at 1 to avoid OOM)
MAX_HOURS    = 8.5    # stop before Kaggle's 9h session limit
STATUS_UPDATE_INTERVAL = 100  # Send status update every N articles (configurable, default 100)

# --- Worker heartbeat and status reporting ---
SESSION_STARTED_AT = datetime.now(timezone.utc)
EMBEDDING_TIMESTAMPS = deque()

# --- Model path (do not change if dataset is attached correctly) ---
matches = glob.glob("/kaggle/input/**/qwen3-embedding", recursive=True)
if matches:
    MODEL_PATH = matches[0]
    print(f"✅ Model found at: {MODEL_PATH}")
else:
    MODEL_PATH = "/kaggle/input/datasets/YOUR_USERNAME/qwen3-embedding-8b-model/qwen3-embedding"
    print(f"⚠️  Model not auto-detected. Set MODEL_PATH manually: {MODEL_PATH}")


def set_n8n_status_url(url):
    global N8N_STATUS_URL
    N8N_STATUS_URL = url
    print(f"✅ N8N Status URL updated: {url[:50]}...")

print(f"\nConfig loaded:")
print(f"  NODE_NAME            : {NODE_NAME}")
print(f"  MODEL_NAME           : {MODEL_NAME}")
print(f"  BATCH_SIZE           : {BATCH_SIZE}")
print(f"  MAX_HOURS            : {MAX_HOURS}h")
print(f"  STATUS_UPDATE_INTERVAL: {STATUS_UPDATE_INTERVAL} articles")
print(f"  N8N_GET              : {N8N_GET_URL[:40]}...")
print(f"  N8N_STATUS           : {N8N_STATUS_URL[:40]}...")
print(f"\nTo change URL at runtime:")
print(f"  set_n8n_status_url('YOUR_NEW_URL')")

In [ ]:
# ============================================================
# CELL 3 — Worker Heartbeat Reporter
# Defines system telemetry, heartbeat payload, and n8n posting logic.
# ============================================================
import requests
import time
from datetime import datetime, timedelta, timezone


def get_server_lan_ip():
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as sock:
            sock.settimeout(0.5)
            sock.connect(("8.8.8.8", 80))
            return sock.getsockname()[0]
    except Exception:
        return "127.0.0.1"


def get_system_metrics():
    vm = psutil.virtual_memory()
    cpu_percent = psutil.cpu_percent(interval=None)
    try:
        load1, load5, load15 = os.getloadavg()
    except Exception:
        load1 = load5 = load15 = 0.0

    try:
        affinity = psutil.Process().cpu_affinity()
        cores_allowed = len(affinity)
        cores_active = cores_allowed
    except Exception:
        cores_allowed = psutil.cpu_count(logical=True) or 0
        cores_active = cores_allowed

    return {
        "server_host": socket.gethostname(),
        "server_lan_ip": get_server_lan_ip(),
        "server_os": platform.system(),
        "server_platform": platform.platform(),
        "cores_logical": psutil.cpu_count(logical=True) or 0,
        "cores_physical": psutil.cpu_count(logical=False) or 0,
        "cores_allowed": cores_allowed,
        "cores_active": cores_active,
        "cpu_percent": round(cpu_percent, 2),
        "load_average_1m": round(load1, 2),
        "load_average_5m": round(load5, 2),
        "load_average_15m": round(load15, 2),
        "memory_total_bytes": vm.total,
        "memory_available_bytes": vm.available,
        "memory_used_percent": round(vm.percent, 2),
    }


def prune_embedding_history():
    cutoff = time.time() - 3600
    while EMBEDDING_TIMESTAMPS and EMBEDDING_TIMESTAMPS[0] < cutoff:
        EMBEDDING_TIMESTAMPS.popleft()


def make_worker_status_payload(
    status,
    cycles,
    articles_fetched,
    articles_embedded,
    articles_errors,
    delay_seconds,
    start_time,
    stop_time=None,
):
    now = datetime.now(timezone.utc)
    uptime = max(time.time() - start_time, 0.0)
    prune_embedding_history()
    embeddings_last_hour = len(EMBEDDING_TIMESTAMPS)
    embeddings_last_hour_per_minute = round(embeddings_last_hour / 60, 2)

    avg_embeddings_per_minute = round(articles_embedded / (uptime / 60), 2) if uptime > 0 else 0.0
    avg_embeddings_per_hour = round(articles_embedded / (uptime / 3600), 2) if uptime > 0 else 0.0

    remainder = STATUS_UPDATE_INTERVAL - (articles_embedded % STATUS_UPDATE_INTERVAL)
    if remainder == STATUS_UPDATE_INTERVAL:
        remainder = STATUS_UPDATE_INTERVAL if articles_embedded > 0 else 0

    next_status_in_seconds = (
        round(remainder / (articles_embedded / uptime), 2)
        if articles_embedded > 0 and uptime > 0
        else 0.0
    )
    next_status_at = (now + timedelta(seconds=next_status_in_seconds)).isoformat()

    system = get_system_metrics()
    device_name = globals().get("device", "unknown")

    return {
        "node_name": NODE_NAME,
        "status": status,
        "cycles": cycles,
        "batch_size": BATCH_SIZE,
        "delay_seconds": delay_seconds,
        "articles_fetched": articles_fetched,
        "articles_embedded": articles_embedded,
        "articles_errors": articles_errors,
        "device": device_name,
        "model_name": MODEL_NAME,
        "server_host": system["server_host"],
        "server_lan_ip": system["server_lan_ip"],
        "server_os": system["server_os"],
        "server_platform": system["server_platform"],
        "session_started_at": SESSION_STARTED_AT.isoformat(),
        "session_uptime_seconds": round(uptime, 2),
        "stop_time": stop_time.isoformat() if stop_time else None,
        "status_interval": STATUS_UPDATE_INTERVAL,
        "next_status_in_seconds": next_status_in_seconds,
        "next_status_at": next_status_at,
        "avg_embeddings_per_hour": avg_embeddings_per_hour,
        "avg_embeddings_per_minute": avg_embeddings_per_minute,
        "embeddings_last_hour": embeddings_last_hour,
        "embeddings_last_hour_per_minute": embeddings_last_hour_per_minute,
        "cores_logical": system["cores_logical"],
        "cores_physical": system["cores_physical"],
        "cores_allowed": system["cores_allowed"],
        "cores_active": system["cores_active"],
        "cpu_percent": system["cpu_percent"],
        "load_average_1m": system["load_average_1m"],
        "load_average_5m": system["load_average_5m"],
        "load_average_15m": system["load_average_15m"],
        "memory_total_bytes": system["memory_total_bytes"],
        "memory_available_bytes": system["memory_available_bytes"],
        "memory_used_percent": system["memory_used_percent"],
    }


def send_worker_status_heartbeat(
    status,
    cycles,
    articles_fetched,
    articles_embedded,
    articles_errors,
    delay_seconds,
    start_time,
    stop_time=None,
):
    if not N8N_STATUS_URL:
        return

    payload = make_worker_status_payload(
        status=status,
        cycles=cycles,
        articles_fetched=articles_fetched,
        articles_embedded=articles_embedded,
        articles_errors=articles_errors,
        delay_seconds=delay_seconds,
        start_time=start_time,
        stop_time=stop_time,
    )

    try:
        resp = requests.post(
            N8N_STATUS_URL,
            headers=N8N_HEADERS,
            json=payload,
            timeout=15,
        )
        if resp.status_code in [200, 201]:
            print("  ✓ Heartbeat sent")
        else:
            print(f"  ⚠️  Heartbeat webhook returned {resp.status_code}")
    except Exception as e:
        print(f"  ⚠️  Failed to send heartbeat: {str(e)[:80]}")


---
## 🤖 Cell 3 — Load Model
Takes 2–3 minutes. Loads the model in GPU float16 (half precision) for best performance on 16 GB GPU memory.

In [ ]:
# ============================================================
# CELL 3 — Load Qwen3-Embedding-8B model on GPU using float16
# ============================================================
from sentence_transformers import SentenceTransformer
import torch
import gc

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required for best performance. Enable GPU accelerator and rerun this notebook.")

device = "cuda"
print(f"Device: {device}")
print(f"GPU memory before load: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")
torch.cuda.empty_cache()

# Load with auto device mapping and float16 offload when possible; fallback to CPU load + GPU half precision.
gc.collect()
load_args = {
    "auto_model_kwargs": {
        "torch_dtype": torch.float16,
        "device_map": "auto",
        "offload_folder": "/tmp/offload",
    },
    "trust_remote_code": True,
}
try:
    model = SentenceTransformer(MODEL_PATH, **load_args)
    print("? Loaded with device_map=auto and float16 offload")
except TypeError:
    print("??  device_map/offload kwargs not supported by SentenceTransformer; falling back to CPU load + GPU half precision.")
    model = SentenceTransformer(MODEL_PATH, device="cpu")
    model = model.half().to(device)

torch.cuda.empty_cache()
used = torch.cuda.memory_allocated(0) / 1e9
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
print(f"GPU used: {used:.1f} GB  |  free: {free:.1f} GB")

# Verify model output dimensions
from numpy import linalg as LA

test_vector = model.encode(
    ["stock market earnings report"],
    normalize_embeddings=False,
    show_progress_bar=False,
    device=device,
    convert_to_numpy=True,
)
if test_vector.ndim == 2:
    test_vector = test_vector / LA.norm(test_vector, axis=1, keepdims=True)

print(f"✅ Model ready — embedding shape: {test_vector.shape}")
if test_vector.shape[1] != 4096:
    print(f"⚠️  WARNING: Expected 4096 dimensions, got {test_vector.shape[1]}")
else:
    print("✅ Dimensions correct: 4096")

---
## 🚀 Cell 4 — Start Embedding
Runs until all articles are embedded or `MAX_HOURS` is reached.

Progress is saved automatically — if the session stops, just run again and it continues from where it left off.

In [ ]:
# ============================================================
# CELL 4 — Main embedding loop
# Fetches articles from n8n → embeds → saves vectors back.
# Safe to stop and resume at any time.
# Progress updates sent to n8n every STATUS_UPDATE_INTERVAL articles.
# ============================================================
import gc
import requests
import time
import torch
import numpy as np
from datetime import datetime, timedelta, timezone

headers = {"X-API-Key": N8N_API_KEY, "Content-Type": "application/json"}
total = 0
errors = 0
start = time.time()
session_start_total = 0
cycles = 0
articles_fetched = 0

print(f"Worker  : {NODE_NAME}")
print(f"Batch   : {BATCH_SIZE}")
print(f"Max time: {MAX_HOURS}h")
print(f"Status updates every: {STATUS_UPDATE_INTERVAL} articles")
print("-" * 80)
print(f"⏱️  Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("-" * 80)

while True:
    cycles += 1

    # --- Time limit ---
    elapsed_h = (time.time() - start) / 3600
    if elapsed_h >= MAX_HOURS:
        print(f"\n{'='*80}")
        print(f"⏱️  Time limit reached ({MAX_HOURS}h). Stopping cleanly.")
        print(f"Total embedded: {total:,}")
        print(f"Session count: {session_start_total:,}")
        print(f"Errors: {errors}")
        print(f"Stopped at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*80}\n")
        send_worker_status_heartbeat(
            status="stopped",
            cycles=cycles,
            articles_fetched=articles_fetched,
            articles_embedded=total,
            articles_errors=errors,
            delay_seconds=0,
            start_time=start,
            stop_time=datetime.now(timezone.utc),
        )
        break

    # --- Get next batch from n8n ---
    try:
        resp = requests.post(
            N8N_GET_URL,
            json={"batch_size": BATCH_SIZE, "node_name": NODE_NAME},
            headers=headers,
            timeout=30,
        )
        records = resp.json().get("records", [])
        articles_fetched += len(records)
    except Exception as e:
        print(f"⚠️  GET error: {e}")
        errors += 1
        time.sleep(10)
        continue

    if not records:
        print(f"\n{'='*80}")
        print(f"✅ No more pending articles. All done!")
        print(f"Total embedded: {total:,}")
        print(f"Session count: {session_start_total:,}")
        print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*80}\n")
        send_worker_status_heartbeat(
            status="done",
            cycles=cycles,
            articles_fetched=articles_fetched,
            articles_embedded=total,
            articles_errors=errors,
            delay_seconds=0,
            start_time=start,
            stop_time=datetime.now(timezone.utc),
        )
        break

    ids = [r["id"] for r in records]
    texts = [r.get("description") or r.get("title") or "" for r in records]

    # --- Embed ---
    try:
        torch.cuda.empty_cache()
        gc.collect()
        with torch.no_grad():
            vectors = model.encode(
                texts,
                batch_size=1,
                normalize_embeddings=False,
                show_progress_bar=False,
                device=device,
                convert_to_numpy=True,
            )
            vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
        torch.cuda.empty_cache()
        gc.collect()
    except RuntimeError as e:
        torch.cuda.empty_cache()
        error_text = str(e).lower()
        if "cuda out of memory" in error_text or "out of memory" in error_text:
            print(f"⚠️  GPU OOM during embed (id={ids}). Clearing cache and retrying.")
            errors += 1
            time.sleep(10)
            torch.cuda.empty_cache()
            continue
        print(f"⚠️  Embed error (id={ids}): {str(e)[:120]}")
        errors += 1
        time.sleep(10)
        continue

    # --- Save vectors to n8n ---
    try:
        payload = {
            "vectors": [
                {"id": ids[i], "vector": vectors[i].tolist(), "node_name": NODE_NAME}
                for i in range(len(ids))
            ]
        }
        requests.post(N8N_SAVE_URL, json=payload, headers=headers, timeout=60)
    except Exception as e:
        print(f"⚠️  SAVE error: {e}")
        errors += 1
        time.sleep(5)
        del vectors
        del payload
        gc.collect()
        torch.cuda.empty_cache()
        continue

    # --- Update counters and heartbeat history ---
    total += len(records)
    session_start_total += len(records)
    EMBEDDING_TIMESTAMPS.extend([time.time()] * len(records))

    del vectors
    del payload
    gc.collect()
    torch.cuda.empty_cache()

    elapsed_s = time.time() - start
    elapsed_m = elapsed_s / 60
    elapsed_h = elapsed_s / 3600

    speed_per_min = total / elapsed_m if elapsed_m > 0 else 0
    speed_per_hour = total / elapsed_h if elapsed_h > 0 else 0
    remaining_h = max(MAX_HOURS - elapsed_h, 0)
    estimated_stop_time = datetime.now(timezone.utc) + timedelta(hours=remaining_h)
    gpu_mem_gb = torch.cuda.memory_allocated(0) / 1e9

    # --- Send status update every N articles ---
    if session_start_total % STATUS_UPDATE_INTERVAL == 0:
        print(f"\n📊 STATUS UPDATE (articles embedded: {session_start_total:,})")
        print(f"  ├─ Total all-time      : {total:,}")
        print(f"  ├─ Speed               : {speed_per_min:.1f} art/min | {speed_per_hour:.1f} art/hour")
        print(f"  ├─ Elapsed time        : {elapsed_m:.1f} min ({elapsed_h:.2f}h)")
        print(f"  ├─ Estimated stop time : {estimated_stop_time.strftime('%H:%M')}")
        print(f"  ├─ GPU memory used     : {gpu_mem_gb:.2f} GB")
        print(f"  └─ Errors              : {errors}")

        send_worker_status_heartbeat(
            status="running",
            cycles=cycles,
            articles_fetched=articles_fetched,
            articles_embedded=total,
            articles_errors=errors,
            delay_seconds=0,
            start_time=start,
        )
        print()

